# **5G Deployment Example with: OAI core and RAN (rfsim) and FlexRIC**

## Install Dependencies:

Install the `ifran` library:

In [ ]:
! pip install ifran

Install the Ansible dependencies:

In [ ]:
! ansible-galaxy install -r ../../ansible/requirements.yaml

Install the SLICES CLI used for the BI to create VMs:

In [ ]:
! pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
! slices auth login

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

In [ ]:
from ifran import MetaModel

metamodel = MetaModel.from_yaml_file("input/metamodel.yaml")
metamodel.to_yaml_file("input/full/metamodel.yaml", exclude_defaults=False)

In [ ]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("input/deployment_model.yaml")
deployment_model.to_yaml_file("input/full/deployment_model.yaml", exclude_defaults=False)

## **1.**   Provisioning:

In [ ]:
from ifran import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [6]:
await prov.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node4' provisioning] ***
changed: [localhost]

TASK [provisioning/bi/provision : Delete experiment with name 'ifran3' if it exists before attempting create for 'node4'] ***
skipping: [localhost]

TASK [provisioning/bi/provision : Wait for experiment deletion to take effect for 'node4'] ***
skipping: [localhost]

TASK [provisioning/bi/provision : Try to create experiment 'ifran3' for 'node4'] ***
skipping: [localhost]

TASK [provisioning/bi/provision : Define SLICES_BI_INFRA and SLICES_EXPERIMENT environment variables for 'node4'] ***
ok: [localhost]

TASK [provisioning/bi/provision : Create 'node4' on infrastructure 'fr-sophia2-bi-vm1' using ex

### **Checkpoint 1:** *Provisiong finished*

In [7]:
metamodel, deployment_model = prov.output()

metamodel.to_yaml_file("output/prov/metamodel.yaml")
deployment_model.to_yaml_file("output/prov/deployment_model.yaml")

## **2.**   Kubernetes deployement:

In [ ]:
from ifran import MetaModel, DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("output/prov/deployment_model.yaml")
metamodel = MetaModel.from_yaml_file("output/prov/metamodel.yaml")

In [ ]:
from ifran import K8SManagement

k8s = K8SManagement(metamodel=metamodel, deployment_model=deployment_model)

In [10]:
await k8s.deploy()


PLAY [node1] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node1'] ****************

PLAY [node2] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node2'] ****************

PLAY [node4] *******************************************************************

PLAY [node3] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node4'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node3'] ****************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1]

TASK [k8

### **Checkpoint 2:** *Kubernetes deployed*

In [11]:
metamodel, deployment_model = k8s.output()

metamodel.to_yaml_file("output/k8s/metamodel.yaml")
deployment_model.to_yaml_file("output/k8s/deployment_model.yaml")

## **3.**   5G core deployement:

In [ ]:
from ifran import MetaModel, DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("output/k8s/deployment_model.yaml")
metamodel = MetaModel.from_yaml_file("output/k8s/metamodel.yaml")

In [ ]:
from ifran import CoreManagement

core = CoreManagement(metamodel=metamodel, deployment_model=deployment_model)

In [14]:
await core.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-database' chart] *****

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-nrf' chart] **********

TASK [helm/prepare : Create Helm build directory for 'oai-udm' chart] **********

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-upf' chart] **********

TASK [helm/prepare : Create Helm build directory for 'oai-amf' chart] **********

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prep

### **Checkpoint 3:** *5G core deployed*

In [15]:
metamodel, deployment_model = core.output()

metamodel.to_yaml_file("output/core/metamodel.yaml")
deployment_model.to_yaml_file("output/core/deployment_model.yaml")

## **4.**   RIC deployement:

In [ ]:
from ifran import MetaModel, DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("output/core/deployment_model.yaml")
metamodel = MetaModel.from_yaml_file("output/core/metamodel.yaml")

In [ ]:
from ifran import RICManagement

ric = RICManagement(metamodel=metamodel, deployment_model=deployment_model)

In [18]:
await ric.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-flexric' chart] ******
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1]

TASK [helm/prepare : Copy local 'oai-ric' Helm chart directory for 'oai-flexric'] ***
changed: [node1]

TASK [helm/prepare : Render 'oai-flexric' Helm Chart.yaml file from Jinja template] ***
changed: [node1]

TASK [helm/prepare : Render templated Helm files for 'oai-flexric'] ************
changed: [node1] => (item=values.yaml)
changed: [node1] => (item=config.yaml)

PLAY RECAP *********************************************************************
node1                      : ok=4    changed=4   

### **Checkpoint 4:** *RIC deployed*

In [19]:
metamodel, deployment_model = ric.output()

metamodel.to_yaml_file("output/ric/metamodel.yaml")
deployment_model.to_yaml_file("output/ric/deployment_model.yaml")

## **5.**   RAN deployement:

In [ ]:
from ifran import MetaModel, DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("output/ric/deployment_model.yaml")
metamodel = MetaModel.from_yaml_file("output/ric/metamodel.yaml")

In [ ]:
from ifran import RANManagement

ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

In [22]:
await ran.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-gnb' chart] **********
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1]

TASK [helm/prepare : Copy local 'oai-gnb' Helm chart directory for 'oai-gnb'] ***
changed: [node1]

TASK [helm/prepare : Render 'oai-gnb' Helm Chart.yaml file from Jinja template] ***
changed: [node1]

TASK [helm/prepare : Render templated Helm files for 'oai-gnb'] ****************
changed: [node1] => (item=values.yaml)
changed: [node1] => (item=config.yaml)

PLAY RECAP *********************************************************************
node1                      : ok=4    changed=4    unreach

### **Checkpoint 5:** *RAN deployed*

In [23]:
metamodel, deployment_model = ran.output()

metamodel.to_yaml_file("output/ran/metamodel.yaml")
deployment_model.to_yaml_file("output/ran/deployment_model.yaml")

## **6.**   UE deployement:

In [ ]:
from ifran import MetaModel, DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("output/ran/deployment_model.yaml")
metamodel = MetaModel.from_yaml_file("output/ran/metamodel.yaml")

In [ ]:
from ifran import UEManagement

ue = UEManagement(metamodel=metamodel, deployment_model=deployment_model)

In [26]:
await ue.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue' chart] *******
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1]

TASK [helm/prepare : Copy local 'oai-containerized' Helm chart directory for 'oai-sim-ue'] ***
changed: [node1]

TASK [helm/prepare : Render 'oai-sim-ue' Helm Chart.yaml file from Jinja template] ***
changed: [node1]

TASK [helm/prepare : Render templated Helm files for 'oai-sim-ue'] *************
changed: [node1] => (item=values.yaml)
changed: [node1] => (item=config.yaml)

PLAY RECAP *********************************************************************
node1                      : ok=4    chan

### **Checkpoint 6:** *UE deployed*

In [27]:
metamodel, deployment_model = ue.output()

metamodel.to_yaml_file("output/ue/metamodel.yaml")
deployment_model.to_yaml_file("output/ue/deployment_model.yaml")

## Cleanup

- #### UE cleanup:

In [ ]:
# from ifran import UEManagement, MetaModel, DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/ue/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/ue/metamodel.yaml")

# ue = UEManagement(metamodel=metamodel, deployment_model=deployment_model)

# await ue.remove(force=True)

- #### RIC cleanup:

In [ ]:
# from ifran import RICManagement, MetaModel, DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/ric/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/ric/metamodel.yaml")

# ric = RICManagement(metamodel=metamodel, deployment_model=deployment_model)

# await ric.remove(force=True)


PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oran-sc-database' Helm chart] ****************

PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oran-sc-chart-repository' Helm chart] ********

PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oran-sc-xapp-onboarder' Helm chart] **********

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oran-sc-a1-mediator' Helm chart] *************

TASK [helm/uninstall : Uninstall 'oran-sc-subscription-manager' Helm chart] ****

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

PLAY [node1] **

- #### RAN cleanup:

In [ ]:
# from ifran import RANManagement, MetaModel, DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/ran/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/ran/metamodel.yaml")

# ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

# await ran.remove(force=True)


PLAY [node1] *******************************************************************

TASK [helm/get_logs : Get logs for 'oai-gnb' Helm chart] ***********************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
[WARNING]: kubernetes<24.2.0 is not supported or tested. Some features may not work.
ok: [node1]

TASK [helm/get_logs : Display logs for 'oai-gnb' Helm chart] *******************
ok: [node1] => {
    "chart_logs.log": "\u001b[NR_MAC] Frame.Slot 256.0\nUE RNTI 9ee8 CU-UE-ID 1 in-sync PH 48 dB PCMAX 20 dBm, average RSRP -43 (16 meas)\nUE 9ee8: dlsch_rounds 3509/0/0/0, dlsch_errors 0, pucch0_DTX 0 (SNR 21.2+1.2 dB), BLER 0.00000 MCS (0) 0 CCE fail 3, goodput 0.00 Mbps\nUE 9ee8: ulsch_rounds 34905/0/0/0, ulsch_errors

- #### 5G core cleanup:

In [ ]:
# from ifran import CoreManagement, MetaModel, DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/core/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/core/metamodel.yaml")

# core = CoreManagement(metamodel=metamodel, deployment_model=deployment_model)

# await core.remove(force=True)

- #### Kubernetes cleanup:

In [ ]:
# from ifran import K8SManagement, MetaModel, DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/k8s/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/k8s/metamodel.yaml")

# k8s = K8SManagement(metamodel=metamodel, deployment_model=deployment_model)

# await k8s.remove(force=True)

- #### Provisioning cleanup:

In [ ]:
# from ifran import ProvisioningManagement, MetaModel, DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/prov/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/prov/metamodel.yaml")

# prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

# await prov.unprovision(force=True)